In [ ]:
import numpy as np

from SFCmodeler.term import Term, Variable, Parameter
from SFCmodeler.SFCmodel import SFCmodel
from SFCmodeler.modelresults import ModelResults

from pysolve3.model import Model

# Model SIM

The `pysolve` version of Godley/Lavoie's model SIM:

In [ ]:
model = Model()

model.set_var_default(0)

model.var('Cd', desc='Consumption goods demand by households')
model.var('Cs', desc='Consumption goods supply')
model.var('Gs', desc='Government goods, supply')
model.var('Hh', desc='Cash money held by households')
model.var('Hs', desc='Cash money supplied by the government')
model.var('Nd', desc='Demand for labor')
model.var('Ns', desc='Supply of labor')
model.var('Td', desc='Taxes, demand')
model.var('Ts', desc='Taxes, supply')
model.var('Y', desc='Income = GDP')
model.var('YD', desc='Disposable income of households')

model.param('Gd', desc='Government goods, demand', default=20.)
model.param('W', desc='Wage rate', default=1.)
model.param('alpha1', desc='Propensity to consume out of income', default=0.6)
model.param('alpha2', desc='Propensity to consume o of wealth', default=0.4)
model.param('theta', desc='Tax rate', default=0.2);

model.add('Cs = Cd')
model.add('Gs = Gd')
model.add('Ts = Td')
model.add('Ns = Nd');
model.add('YD = (W*Ns) - Ts');
model.add('Td = theta * W * Ns');
model.add('Cd = alpha1*YD + alpha2*Hh(-1)');
model.add('Hs - Hs(-1) =  Gd - Td');
model.add('Hh - Hh(-1) = YD - Cd');
model.add('Y = Cs + Gs');
model.add('Nd = Y/W');

for _ in range(100):
    model.solve(iterations=100, threshold=1e-10);

psSIMresult = ModelResults(model.solutions)

The `SFCmodeler` version of model SIM:

In [ ]:
Variable.default_value = 0

Cd = Variable("Cd", 'Consumption goods demand by households')
Cs = Variable("Cs", 'Consumption goods supply')
Gs = Variable("Gs", 'Government goods, supply')
Hh = Variable("Hh", 'Cash money held by households')
Hs = Variable("Hs", 'Cash money supplied by the government')
Nd = Variable("Nd", 'Demand for labor')
Ns = Variable("Ns", 'Supply of labor')
Td = Variable("Td", 'Taxes, demand')
Ts = Variable("Ts", 'Taxes, supply')
Y = Variable("Y", 'Income = GDP')
YD = Variable("YD", 'Disposable income of households')

Gd = Parameter("Gd",'Government goods, demand', 20.)
W = Parameter("W",'Wage rate', 1.)
alpha1 = Parameter("alpha1",'Propensity to consume out of income', 0.6)
alpha2 = Parameter("alpha2",'Propensity to consume o of wealth', 0.4)
theta = Parameter("theta",'Tax rate', 0.2);

model = SFCmodel([
  Cs << Cd,
  Gs << Gd,
  Ts << Td,
  Ns << Nd,
  YD << (W*Ns) - Ts,
  Td << theta * W * Ns,
  Cd << alpha1*YD + alpha2*Hh(-1),
  Hs <<  Hs(-1) + Gd - Td,
  Hh << Hh(-1) + YD - Cd,
  Y << Cs + Gs,
  Nd << Y/W,
])

ssSIMresult = model.solve(100)

  0%|          | 0/101 [00:00<?, ?it/s]

Check that all endogenous variables produce equal results (within solver error):

In [ ]:
for var in model.variables:
    assert np.isclose(psSIMresult.df[str(var)].iloc[-1], ssSIMresult[var][-1],rtol=1e-4).all()
    #pysolve algorithm doesn't quite get exactly to steady state, so need larger rtol than default

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()